# Tusker: Qwen3 embeddings on Colab

A thin driver over `scripts/embed_corpus.py`, which also runs on a laptop or a Kaggle
kernel. All the logic lives there.

## Before you start, on your laptop

1. Confirm the articles are ingested. `article_embeddings.article_id` is a foreign key
   to `articles.id`, and embeddings with no matching article are skipped and logged
   rather than raised, so a finished run can silently load nothing.
   ```bash
   docker compose exec -T pg-news-native psql -U news_user -d news_db -c "select count(*) from articles;"
   ```
2. Shrink the corpus, if you have not already.
   ```bash
   uv run scripts/slim_corpus.py datasets/cc-news/canonical-dataset.jsonl
   ```
3. Put both files in Drive under `MyDrive/tusker/`: the `.jsonl.gz` and
   `scripts/embed_corpus.py`. The script rides with the data, so this notebook does not
   depend on the repo being pushed. `rclone copy` or Google Drive for Desktop both work.

Set Runtime, Change runtime type, GPU before running anything below.

## 1. Dependencies

Colab ships torch, numpy and pyarrow. If the transformers pin conflicts with what is
preinstalled, or pip asks for a runtime restart, drop the version and use what is there.
The vectors depend on the model weights and the pooling code in `embed_corpus.py`, not
on the transformers version.

In [ ]:
%%capture
!pip install -q transformers==4.51.3

## 2. Mount Drive and check the payload

Peak usage is the checkpoints plus the merged Parquet. Step 3 passes
`--clean-checkpoints` so the shards go once the merge succeeds, but the peak still has
to fit in free Drive space. Running out at 70% is the worst way for this to fail.

In [ ]:
import gzip
import os

from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/tusker"
SCRIPT  = f"{PROJECT}/embed_corpus.py"
CORPUS  = f"{PROJECT}/canonical-dataset-slim.jsonl.gz"
OUTPUT  = f"{PROJECT}/cc-news-embeddings.parquet"

# On /content these vanish with the runtime, which defeats the point of checkpointing.
CHECKPOINTS = f"{PROJECT}/checkpoints"

for path in (SCRIPT, CORPUS):
    if not os.path.exists(path):
        raise FileNotFoundError(f"missing on Drive: {path}")

with gzip.open(CORPUS, "rb") as fh:
    docs = sum(chunk.count(b"\n") for chunk in iter(lambda: fh.read(8 << 20), b""))
vectors_gb = docs * 1024 * 4 / 1e9

print(f"corpus        : {CORPUS}  ({os.path.getsize(CORPUS) / 1e6:.1f} MB, {docs:,} docs)")
print(f"output        : {OUTPUT}")
print(f"peak on Drive : ~{vectors_gb * 2:.2f} GB (checkpoints + parquet)")
print("\nCheck your free Drive space against that peak before continuing.")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3. Run

`--auto-batch-size` times a 2000-doc slice at 64, 256 and 512 and uses the fastest. The
default of 64 was tuned on Apple Silicon and is usually wrong for a discrete GPU.

If the session drops partway, run this cell again: finished shards are skipped and it
picks up where it stopped. Once the run has completed, re-running prints "already
complete" and does nothing; add `--force` to regenerate.

In [ ]:
!python "{SCRIPT}" "{CORPUS}" \
    --out "{OUTPUT}" \
    --checkpoint-dir "{CHECKPOINTS}" \
    --auto-batch-size \
    --clean-checkpoints

## 4. Bring it home

`cc-news-embeddings.parquet` is on Drive. Pull it down the way you pushed the corpus up,
into `datasets/cc-news/`, then load it. No object store is involved:

```bash
cp cmd/datapipe/embeddings.env.example cmd/datapipe/embeddings.env
# EMBEDDING_SOURCE=file
# EMBEDDING_FILE_PATH=./datasets/cc-news/cc-news-embeddings.parquet
make run-datapipe-embeddings-pg
```

The `model` key in the file metadata becomes `article_embeddings.model_name`, so it
matches the query-time model without extra configuration.